 # Notebook 08 - RHI scanning dual-frequency LWC retrieval



 Implementation follows Lamer, Tatarevic, Jo & Kollias (2014),

 *Evaluation of gridded scanning ARM cloud radar reflectivity observations

 and vertical Doppler velocity retrievals*, AMT 7, 1089-1103.



 ## Why the retrieval is done in POLAR coordinates



 Lamer Sect. 5 (Summary), final paragraph:



   "one must consider that the gridding of the radar observables (Doppler

    velocity and radar reflectivity) is recommended for the purpose of

    investigating the clouds' 3-D morphology and dynamical structure.

    Microphysical retrievals, especially those based on dual-wavelength

    measurements available from the SACRs, should be performed at the SACR

    native coordinate system since small changes in the SACR observables due

    to the gridding can have a large effect on retrievals."



 So the order here is:

     polar Ze -> polar DWR -> polar dDWR/ds -> polar LWC -> grid the LWC



 The gridding (Sect. 3.3, Appendix A) is applied only at the end, for display

 and for comparison with other fields.



 Save as notebooks_<DATASET>/08_rhi_scanning_retrieval.ipynb

In [ ]:
import sys
from pathlib import Path

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

sys.path.append(str(Path("../src").resolve()))
from radar_lwc.retrieval.zhu import retrieve_lwc_zhu

Processed_Dataset = "20230930_PPI"
PROC = Path(f"../data/processed/{Processed_Dataset}")

# --- instrument -----------------------------------------------------------
HPBW_KA_DEG = 0.84
HPBW_W_DEG  = 0.55
Z_RADAR_M   = 0.0                 # antenna height above grid origin
EARTH_RADIUS_M = 6_371_000.0      # Lamer Eq. (1)

# --- scan geometry --------------------------------------------------------
# Lamer Sect. 3.3.1: "the angle difference between two successive elevations
# is equal to the antenna 3 degree beamwidth". With two beamwidths, use the
# SMALLER one so neither radar is undersampled.
ELEV_STEP_DEG = HPBW_W_DEG

# --- grid (user-dependent, Lamer Sect. 3.3) -------------------------------
# Lamer Fig. 5 uses dx = dz = 50 m.
DX_M, DZ_M = 50.0, 50.0
X_MIN_M, X_MAX_M = -2000.0, 2000.0
Z_MIN_M, Z_MAX_M =     0.0, 3000.0

# --- retrieval ------------------------------------------------------------
N_DEFAULT       = 9
MIN_CLOUD_GATES = 3
EDGE_PAD        = 1
GRID_METHOD     = "cressman"      # "cressman" (App. A3) or "maximum" (App. A1)



In [ ]:
# ---- scan configuration -------------------------------------------------
SCAN_TYPE = "ppi"          # "zenith" | "ppi" | "rhi"

if SCAN_TYPE == "zenith":
    Dataset_94GHz     = "20230930_94GHz"
    Dataset_35GHz     = "20230930_35GHz"
    Processed_Dataset = "20230930"
    SCAN_TOKEN        = "ZEN"
    ELEV_DEG          = 90.0
    TIME_AVERAGE_S    = 30.0          # rolling mean over profiles is legal
else:
    Dataset_94GHz     = "20230930_94GHz_PPI"
    Dataset_35GHz     = "20230930_35GHz_PPI"
    Processed_Dataset = "20230930_PPI"
    SCAN_TOKEN        = "PPI"         # <-- replace with the answer from 0.1
    ELEV_DEG          = 75.0          # <-- replace with the answer from 0.2
    TIME_AVERAGE_S    = 0.0           # each ray stands alone

SIN_ELEV = np.sin(np.radians(ELEV_DEG))
print(f"SCAN_TYPE={SCAN_TYPE}  elev={ELEV_DEG} deg  sin(elev)={SIN_ELEV:.4f}")

 ## Step 1 - Load the scan data



 Same file structure as the zenith pipeline, `_scan` suffix. Expected

 variables on dims (Time, range), where `Time` now indexes RAYS and `range`

 is SLANT range:



   Ze_W, Ze_Ka, DWR, common_mask, alpha_W, alpha_Ka, kappa_W, kappa_Ka,

   T_cloud, SLv_W, SLv_Ka, LDR_Ka



 plus per-ray `elevation` and `azimuth` in degrees.

In [ ]:
suffix = "" if SCAN_TYPE == "zenith" else "_scan"
ds_retrieval_scan = xr.open_dataset(
    PROC / f"retrieval_input_4km_alpha_kappa_calculated{suffix}.nc")

TR      = lambda a: a.transpose("Time", "range")
target  = TR(ds_retrieval_scan["DWR"])
to_grid = lambda a: TR(a).reindex_like(target)

rng_m    = target["range"].values.astype(float)     # SLANT range
elev_deg = ds_retrieval_scan["elevation"].values
azi_deg  = ds_retrieval_scan["azimuth"].values

n_ray, n_gate = target.shape
dr_m = float(np.median(np.diff(rng_m)))

print(f"{n_ray} rays x {n_gate} gates")
print(f"slant range   {rng_m[0]:.0f}-{rng_m[-1]:.0f} m, {dr_m:.1f} m gates")
print(f"elevation     {elev_deg.min():.1f}-{elev_deg.max():.1f} deg")
print(f"azimuth       {np.median(azi_deg):.1f} deg "
      f"(spread {azi_deg.max()-azi_deg.min():.2f})")
print(f"elev step     {np.median(np.abs(np.diff(elev_deg))):.3f} deg "
      f"(target {ELEV_STEP_DEG:.2f})")

# Lamer Sect. 3.3.1: gaps between adjacent elevations "seldom wider than two
# times" the beamwidth. Flag rays that violate this.
gap = np.abs(np.diff(elev_deg))
print(f"rays with gap > 2x beamwidth: {(gap > 2*ELEV_STEP_DEG).sum()} "
      f"of {len(gap)}")



 ## Step 2 - Polar to Cartesian coordinates



 Lamer Sect. 3.3.1, Eqs. (1)-(5), 4/3 effective Earth radius beam propagation

 model (Doviak and Zrnic, 1993):



     a_ef = 4/3 a

     z    = (r^2 + a_ef^2 + 2 r a_ef sin(theta))^0.5 - a_ef

     s    = a_ef asin( r cos(theta) / (a_ef + z) )

     x    = s cos(phi),   y = s sin(phi)



 "Setting the azimuth angle to 0 degrees performs the transition from 3-D to

 the range-height plane imposed by the CW scanning strategy."



 z is needed BEFORE the retrieval, because the sounding-derived alpha, kappa

 and T must be sampled at the HEIGHT of each polar bin, not at slant range.

In [ ]:
a_ef = (4.0 / 3.0) * EARTH_RADIUS_M
th   = np.radians(elev_deg)[:, None]                  # (ray, 1)
rr   = rng_m[None, :]                                 # (1, gate)

z_obs = np.sqrt(rr**2 + a_ef**2 + 2.0 * rr * a_ef * np.sin(th)) - a_ef
z_obs = z_obs + Z_RADAR_M                                    # height [m]
x_obs = a_ef * np.arcsin(rr * np.cos(th) / (a_ef + z_obs))   # along-scan [m]

print(f"heights    {np.nanmin(z_obs):.0f}-{np.nanmax(z_obs):.0f} m")
print(f"along-scan {np.nanmin(x_obs):.0f}-{np.nanmax(x_obs):.0f} m")

# sanity: alpha comes from the sounding, so it must track HEIGHT, not gate index
alpha_w_v = to_grid(ds_retrieval_scan["alpha_W"]).values
k_lo, k_hi = n_gate // 4, n_gate // 2
print(f"alpha_W  ray 0,   gate {k_lo}: z={z_obs[0, k_lo]:6.0f} m -> "
      f"{alpha_w_v[0, k_lo]:.3f} dB/km")
print(f"alpha_W  ray -1,  gate {k_hi}: z={z_obs[-1, k_hi]:6.0f} m -> "
      f"{alpha_w_v[-1, k_hi]:.3f} dB/km")
print("  -> should follow HEIGHT; if it follows gate index the sounding was "
      "not projected onto the slant line-of-sight")



 ## Step 3 - Post-processing in polar coordinates



 Lamer Sect. 2, in order:

   (i)   feature mask - receiver noise + power threshold, then a

         range-elevation spatial filter (Clothiaux et al. 1995;

         Hildebrand and Sekhon 1974)

   (ii)  gaseous attenuation from the nearest sounding, projected onto the

         slant line-of-sight (Rosenkranz 1998)

   (iii) Doppler velocity unfolding - not needed here, reflectivity only

   (iv)  insect filter, applied where cloud base is detected within a 1 h

         window and T > 5 C, using LDR to separate insects from cloud



 (ii) is handled inside the LWC equation through the Delta_alpha term,

 exactly as in the zenith pipeline, so Ze is NOT corrected here - doing both

 would double-count the gas.

In [ ]:
SNR_MARGIN_DB = 3.0

ze_ka = to_grid(ds_retrieval_scan["Ze_Ka"]).values
ze_w  = to_grid(ds_retrieval_scan["Ze_W"]).values
sl_ka = 10.0 * np.log10(to_grid(ds_retrieval_scan["SLv_Ka"]).values)
sl_w  = 10.0 * np.log10(to_grid(ds_retrieval_scan["SLv_W"]).values)

mask_scan = (np.isfinite(ze_ka) & np.isfinite(ze_w)
             & (ze_ka > sl_ka + SNR_MARGIN_DB)
             & (ze_w  > sl_w  + SNR_MARGIN_DB))
n0 = int(mask_scan.sum())

# range-elevation spatial filter: keep bins with at least 2 detected neighbours
nb = np.zeros_like(mask_scan, dtype=int)
nb[1:, :]  += mask_scan[:-1, :]
nb[:-1, :] += mask_scan[1:, :]
nb[:, 1:]  += mask_scan[:, :-1]
nb[:, :-1] += mask_scan[:, 1:]
mask_scan &= (nb >= 2)
print(f"(i)  feature mask: {n0} -> {int(mask_scan.sum())} bins after filter")

# (iv) insect filter
T_C = to_grid(ds_retrieval_scan["T_cloud"]).values - 273.15
LDR_THRESHOLD_DB = -20.0
if "LDR_Ka" in ds_retrieval_scan:
    ldr = to_grid(ds_retrieval_scan["LDR_Ka"]).values
    insect = (T_C > 5.0) & np.isfinite(ldr) & (ldr > LDR_THRESHOLD_DB)
    print(f"(iv) LDR insect filter: removed {int(insect.sum())} bins")
else:
    # fallback: insects are non-Rayleigh at W-band and give strongly negative
    # DWR with no height dependence
    insect = (T_C > 5.0) & (target.values < -1.0)
    print(f"(iv) no LDR, DWR < -1 dB fallback: removed {int(insect.sum())} bins")
mask_scan &= ~insect

print(f"final cloudy bins: {int(mask_scan.sum())} ({mask_scan.mean():.1%})")
mask_scan_da = xr.DataArray(mask_scan, dims=target.dims, coords=target.coords)



 ## Step 4 - LWC retrieval in native polar coordinates



 `retrieve_lwc_zhu` is reused unchanged. It differentiates DWR against

 whatever sits in the `range` coordinate; here that is SLANT range, so it

 returns dDWR/ds. That is the correct quantity - attenuation accumulates

 along the propagation path, not along the vertical.



     LWC = (0.5 dDWR/ds - Delta_alpha) / Delta_kappa



 Heights come from z_obs (Step 2) and are attached afterwards.

In [ ]:
lwc_polar, ddwr_ds, m_eff, n_windows, n_used = retrieve_lwc_zhu(
    dwr        = target,
    kappa_w    = to_grid(ds_retrieval_scan["kappa_W"]),
    kappa_ka   = to_grid(ds_retrieval_scan["kappa_Ka"]),
    alpha_w    = to_grid(ds_retrieval_scan["alpha_W"]),
    alpha_ka   = to_grid(ds_retrieval_scan["alpha_Ka"]),
    cloud_mask = mask_scan_da,
    N_default  = N_DEFAULT,
    min_cloud_gates = MIN_CLOUD_GATES,
    edge_pad   = EDGE_PAD,
)

lwc_v = lwc_polar.where(mask_scan_da).values
print(f"retrieved bins {int(np.isfinite(lwc_v).sum())}")
print(f"LWC median {np.nanmedian(lwc_v):+.3f}, std {np.nanstd(lwc_v):.3f} g/m3")
print(f"N used: median {float(n_used.where(n_used>0).median()):.0f}, "
      f"min {int(n_used.where(n_used>0).min())}, max {int(n_used.max())}")



 ## Step 5 - Grid to Cartesian



 Lamer Appendix A, Eqs. (A3)-(A6):



     R    = max(Rd1, Rd2)                                            (A5)

     Rd1  = sqrt( (dx^2 + dz^2) / 4 )                                (A6)

     Rd2  = sqrt( dr^2 + (r + 0.5 dr)^2 sin^2(0.5 max(dtheta, bw)) ) (A6)

     w    = (R^2 - d^2) / (R^2 + d^2)   for d^2 <= R^2, else 0       (A4)

     Fg   = sum(w F) / sum(w)                                        (A3)



 Appendix A: "only cloud pixels are examined when selecting the observational

 values that 'influence' the value at a specific grid point."



 Lamer Sect. 5 also notes a hybrid option - Cressman everywhere except near

 cloud edges, where the maximum value scheme (A1) performs best "since the

 quality of the radar observables strongly depends on signal-to-noise

 conditions". Both are available via GRID_METHOD.

In [ ]:
grid_x = np.arange(X_MIN_M, X_MAX_M + DX_M, DX_M)
grid_z = np.arange(Z_MIN_M, Z_MAX_M + DZ_M, DZ_M)

dtheta_deg = float(np.median(np.abs(np.diff(elev_deg))))
ang        = np.radians(0.5 * max(dtheta_deg, ELEV_STEP_DEG))
Rd1        = np.sqrt((DX_M**2 + DZ_M**2) / 4.0)
Rd2        = np.sqrt(dr_m**2 + (rng_m + 0.5 * dr_m)**2 * np.sin(ang)**2)
R_infl     = np.maximum(Rd1, Rd2)                      # (gate,)

print(f"Rd1 = {Rd1:.1f} m (grid)")
print(f"Rd2 = {Rd2.min():.1f}-{Rd2.max():.1f} m (radar volume)")
print(f"R   = {R_infl.min():.1f}-{R_infl.max():.1f} m")


def grid_polar(field_polar, valid_polar, method=GRID_METHOD):
    """Lamer Appendix A1 (maximum) / A3-A4 (Cressman)."""
    nx, nz = len(grid_x), len(grid_z)
    num = np.zeros((nx, nz))
    den = np.zeros((nx, nz))
    out = np.full((nx, nz), -np.inf)

    for i in range(n_ray):
        for k in range(n_gate):
            if not valid_polar[i, k] or not np.isfinite(field_polar[i, k]):
                continue
            R = R_infl[k]
            xo, zo = x_obs[i, k], z_obs[i, k]
            ix = np.flatnonzero(np.abs(grid_x - xo) <= R)
            iz = np.flatnonzero(np.abs(grid_z - zo) <= R)
            if ix.size == 0 or iz.size == 0:
                continue
            d2 = ((grid_x[ix][:, None] - xo)**2
                  + (grid_z[iz][None, :] - zo)**2)
            inside = d2 <= R**2
            if not inside.any():
                continue
            if method == "cressman":
                w = np.where(inside, (R**2 - d2) / (R**2 + d2), 0.0)      # (A4)
                num[np.ix_(ix, iz)] += w * field_polar[i, k]
                den[np.ix_(ix, iz)] += w
            else:
                blk = out[np.ix_(ix, iz)]
                out[np.ix_(ix, iz)] = np.where(
                    inside, np.maximum(blk, field_polar[i, k]), blk)      # (A1)

    if method == "cressman":
        return np.where(den > 0, num / den, np.nan)                       # (A3)
    return np.where(np.isfinite(out), out, np.nan)


valid_lwc = mask_scan & np.isfinite(lwc_v)

lwc_grid = grid_polar(lwc_v, valid_lwc)
dwr_grid = grid_polar(target.values, mask_scan)
ze_grid  = grid_polar(ze_ka, mask_scan)

print(f"grid {len(grid_x)} x {len(grid_z)}, "
      f"{int(np.isfinite(lwc_grid).sum())} filled LWC cells")



 ## Step 6 - Domain limits



 Lamer Sect. 3.1: sensitivity falls as r^2, and for non-drizzling

 stratocumulus "a drop of the radar sensitivity from -30 to -20 dBZ can

 result in a dramatic change in the radar-derived CF_Hor" (8 % per 1 dBZ).

 In their stratus case the cloud "is not detected at ranges extending farther

 than 5 km from the Ka-SACR".



 Sect. 4.2: for the stratus layer the retrieval standard deviation "is almost

 constant within +/- 1.5 km along the scan range from the radar. Beyond this

 range, the standard deviation grows fast."



 Sect. 5: CW-RHI can give robust representations "up to 25-30 degrees off

 zenith".

In [ ]:
off_zenith = np.repeat(np.abs(90.0 - elev_deg)[:, None], n_gate, axis=1)

print(f"{'|x| bin':>12}{'bins':>8}{'LWC std':>10}{'Ze med':>9}{'off-zen':>9}")
for lo in range(0, 2500, 500):
    sel = mask_scan & (np.abs(x_obs) >= lo) & (np.abs(x_obs) < lo + 500)
    if sel.sum() < 50:
        continue
    v = lwc_v[sel]; v = v[np.isfinite(v)]
    if v.size < 20:
        continue
    print(f"{lo:>5}-{lo+500:<6}{int(sel.sum()):>8}{np.std(v):>10.3f}"
          f"{np.nanmedian(ze_ka[sel]):>9.1f}{np.median(off_zenith[sel]):>9.1f}")
print("\nLamer: stratus stable to |x| ~ 1.5 km and 25-30 deg off zenith")



 ## Step 7 - Plot

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(10, 10), sharex=True, sharey=True)

p0 = ax[0].pcolormesh(grid_x / 1000, grid_z, ze_grid.T,
                      cmap="viridis", shading="auto")
plt.colorbar(p0, ax=ax[0], label="Ze$_{Ka}$ [dBZ]")
ax[0].set_ylabel("height [m]")
ax[0].set_title(f"RHI, {GRID_METHOD} gridding "
                f"(dx={DX_M:.0f} m, dz={DZ_M:.0f} m)")

p1 = ax[1].pcolormesh(grid_x / 1000, grid_z, dwr_grid.T,
                      cmap="RdBu_r", shading="auto", vmin=-2, vmax=4)
plt.colorbar(p1, ax=ax[1], label="DWR [dB]")
ax[1].set_ylabel("height [m]")

p2 = ax[2].pcolormesh(grid_x / 1000, grid_z, lwc_grid.T,
                      cmap="Blues", shading="auto", vmin=0, vmax=0.6)
plt.colorbar(p2, ax=ax[2], label="LWC [g m$^{-3}$]")
ax[2].set_ylabel("height [m]")
ax[2].set_xlabel("along-scan distance [km]")

for a in ax:
    a.axvline(-1.5, color="k", ls=":", lw=1)
    a.axvline(+1.5, color="k", ls=":", lw=1)

plt.tight_layout()
plt.show()



 ## Step 8 - Save

In [ ]:
ds_scan_out = xr.Dataset(
    {
        "LWC":        (("x", "z"), lwc_grid),
        "DWR":        (("x", "z"), dwr_grid),
        "Ze_Ka":      (("x", "z"), ze_grid),
        "LWC_polar":  (("Time", "range"), lwc_v),
        "DWR_polar":  (("Time", "range"), target.values),
        "mask_polar": (("Time", "range"), mask_scan),
        "height":     (("Time", "range"), z_obs),
        "along_scan": (("Time", "range"), x_obs),
        "N_used":     (("Time",), n_used.values),
        "elevation":  (("Time",), elev_deg),
    },
    coords={"x": grid_x, "z": grid_z,
            "Time": target["Time"], "range": rng_m},
    attrs={
        "reference": "Lamer et al. (2014), AMT 7, 1089-1103",
        "gridding": f"{GRID_METHOD}, Appendix A",
        "retrieval_coords": ("native polar, per Lamer Sect. 5: dual-wavelength "
                             "microphysical retrievals should be performed in "
                             "the native coordinate system"),
        "dx_m": DX_M, "dz_m": DZ_M,
        "elev_step_deg": dtheta_deg,
        "hpbw_ka_deg": HPBW_KA_DEG, "hpbw_w_deg": HPBW_W_DEG,
        "N_default": N_DEFAULT, "edge_pad": EDGE_PAD,
        "snr_margin_db": SNR_MARGIN_DB,
    },
)
out_path = PROC / "rhi_lwc_gridded_scan.nc"
ds_scan_out.to_netcdf(out_path)
print(f"saved -> {out_path}")



 ## Step 9 (later) - Along-time gridding



 Lamer Sect. 3.3.2. Only needed once several consecutive RHI scans are

 combined into a set.



   "The selected temporal resolution dt is preferred to be greater than or

    equal to the average time between two successive scans ... Furthermore,

    dt should not be higher than a quarter of the scan duration."



 Nearest-neighbour along time populates the "measured values", then:

   case 1 - both bracketing measured values contain radar detections

            -> linear interpolation for all points between them

   case 2 - one of the two contains no detection

            -> each measured value is held constant and propagated over half

               the interval, i.e. only to the start time of the following

               and/or the end time of the preceding scan

In [ ]:
# SCAN_DURATION_S = ...      # horizon-to-horizon
# DT_S = ...                 # >= mean gap between scans, <= SCAN_DURATION_S / 4
